In [ ]:
# [BLOCK 1] CÀI ĐẶT THƯ VIỆN & KẾT NỐI API

# 1. Cài đặt chế độ ẩn (-q) các thư viện lõi: pinecone (lưu trữ vector), transformers (chạy mô hình AI), pandas (xử lý dữ liệu), google-genai (giao tiếp Gemini API), nltk và underthesea (xử lý ngôn ngữ tự nhiên), rapidfuzz (so khớp chuỗi), và deep-translator (dịch văn bản).
!pip install -q pinecone transformers pandas google-genai nltk underthesea rapidfuzz deep-translator

# 2. Import các thư viện hệ thống và xử lý mảng/bảng dữ liệu cơ bản
import os, re, csv, shutil, json, time
import pandas as pd
import numpy as np

# 3. Import Pytorch (để chạy model SigLIP trên GPU) và PIL (để mở, xử lý hình ảnh)
import torch
from PIL import Image

# 4. Cài đặt các công cụ tối ưu tốc độ: lru_cache (Lưu nháp bộ nhớ tạm) và ThreadPoolExecutor (Chạy đa luồng song song)
from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor, as_completed

# 5. Import SDK của các nền tảng AI đám mây (Google Gemini, Pinecone, HuggingFace)
from google import genai
from google.genai import types
from pinecone import Pinecone
from transformers import AutoModel, AutoTokenizer
from deep_translator import GoogleTranslator

# 6. Công cụ tải file xuống máy tính từ Colab
from google.colab import files

# 7. Khai báo API Key để cấp quyền truy cập vào Pinecone và Gemini
PINECONE_API_KEY = "YOUR_API_KEY"
GEMINI_API_KEY = "YOUR_API_KEY"

# 8. Xin quyền và kết nối vào Google Drive (để code đọc được ảnh và file Map Keyframe)
from google.colab import drive
drive.mount('/content/drive')

# 9. Đăng nhập vào hệ thống Pinecone và trỏ thẳng vào dataset có tên "aic-26"
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("aic-26")

# 10. Khởi tạo client kết nối với Gemini API để chuẩn bị gọi mô hình Ngôn ngữ - Hình ảnh (VLM) cho bước xác thực.
client = genai.Client(api_key=GEMINI_API_KEY)

# 11. Tự động kiểm tra môi trường thực thi: thiết lập thiết bị xử lý là GPU ("cuda") nếu hệ thống có hỗ trợ, ngược lại sẽ sử dụng CPU ("cpu").
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.4 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
# [BLOCK 2] NẠP MODEL LÊN GPU & HỆ THỐNG MAP KEYFRAME

# 1. Khai báo định danh của mô hình SigLIP (mô hình ngôn ngữ - hình ảnh đa phương thức) trên HuggingFace.
MODEL_NAME = "google/siglip-so400m-patch14-384"

# 2. Khởi tạo tokenizer để xử lý và mã hóa dữ liệu văn bản đầu vào thành các token.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 3. Tải trọng số mô hình SigLIP vào bộ nhớ.
# Tự động ép kiểu dữ liệu về float16 (FP16) nếu sử dụng GPU để tối ưu VRAM và tăng tốc độ suy luận, hoặc float32 nếu dùng CPU.
# Phương thức .eval() đặt mô hình vào chế độ suy luận (tắt các chức năng training như Dropout).
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE.type == 'cuda' else torch.float32
).text_model.to(DEVICE).eval()

# 4. Khởi tạo công cụ dịch thuật để chuyển đổi câu truy vấn từ tiếng Việt sang tiếng Anh nhằm tối ưu độ chính xác của mô hình SigLIP.
translator = GoogleTranslator(source='vi', target='en')

# 5. Khai báo đường dẫn trỏ tới thư mục chứa các tệp CSV ánh xạ khung hình (Map Keyframe) trên Google Drive.
MAP_DIR = "/content/drive/MyDrive/AIC26/mapkeyframes"

# 6. Khởi tạo và kiểm tra từ điển toàn cục (global dictionary) 'map_db'.
# Việc này giúp bộ nhớ đệm (cache) giữ lại dữ liệu ánh xạ, tránh tình trạng đọc lại file từ đầu mỗi khi thực thi lại block code.
if 'map_db' not in globals() or not map_db:
    map_db = {}

    if os.path.exists(MAP_DIR):
        # 7. Duyệt qua tất cả các tập tin trong thư mục, chỉ lọc và xử lý các tập tin có định dạng .csv
        for file_name in os.listdir(MAP_DIR):
            if not file_name.endswith('.csv'): continue

            # 8. Trích xuất mã định danh của video (Video ID) bằng cách loại bỏ đuôi '.csv' và các khoảng trắng thừa.
            vid = file_name.replace('.csv', '').strip().upper()

            try:
                # 9. Đọc tập tin CSV và kiểm tra tính toàn vẹn của dữ liệu thông qua hai cột bắt buộc: 'n' (số thứ tự keyframe) và 'frame_idx' (số khung hình gốc).
                df_map = pd.read_csv(os.path.join(MAP_DIR, file_name))
                if 'n' in df_map.columns and 'frame_idx' in df_map.columns:

                    # 10. Chuyển đổi cấu trúc bảng thành từ điển và lưu vào map_db với cấu trúc: map_db[video_id] = {n: frame_idx}
                    map_db[vid] = {int(row['n']): int(row['frame_idx']) for _, row in df_map.iterrows()}
            except Exception as e:
                pass

config.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  798kB            

spiece.model: downloading bytes:           |  0.00B            

...

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.51GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

In [3]:
# [BLOCK 3] LÕI THUẬT TOÁN (TRUY XUẤT VECTOR & QUY HOẠCH ĐỘNG)

# Sử dụng @lru_cache để lưu bộ nhớ đệm (tối đa 128 kết quả), giúp giảm độ trễ và tiết kiệm API quota khi gặp lại các truy vấn trùng lặp.
@lru_cache(maxsize=128)
def quest_rewrite(query_text):
    prompt = f"""Bạn là chuyên gia thị giác máy tính. Chuyển đổi truy vấn sau thành chuỗi từ khóa miêu tả hình ảnh (Tiếng Anh). Bỏ từ trừu tượng (đầu tiên, chuẩn bị). Trọng tâm vào vật thể chạm nhau. Truy vấn: '{query_text}'"""
    try:
        # Gọi mô hình LLM để tối ưu hóa câu truy vấn thành các từ khóa thị giác.
        res = client.models.generate_content(
            model='gemini-3.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0)
        )
        return res.text.strip()
    except: return query_text

def aic_retrieval_engine(query_vi, use_quest=True, top_k_pinecone=1000, top_k_final=400):
    try: en_query = translator.translate(query_vi)
    except: en_query = query_vi

    visual_query = quest_rewrite(en_query) if use_quest else en_query

    # Mã hóa chuỗi văn bản và trích xuất đặc trưng vector (Text Embedding) thông qua mô hình SigLIP.
    inputs = tokenizer([visual_query], padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        query_vector = model(**inputs).pooler_output[0].cpu().numpy().tolist()

    # Truy vấn độ tương đồng Cosine trên cơ sở dữ liệu Pinecone để tìm kiếm các vector không gian gần nhất.
    pc_results = index.query(vector=query_vector, top_k=top_k_pinecone, include_metadata=True)['matches']

    # Sắp xếp và định dạng lại kết quả trả về, đồng thời trích xuất ID video và chỉ số keyframe (n) từ siêu dữ liệu (metadata).
    return sorted([{"video_id": m.get('metadata', {}).get('video', '').strip().upper(), "n": int(''.join(filter(str.isdigit, m.get('metadata', {}).get('frame', '0'))) or 0), "score": m['score']} for m in pc_results], key=lambda x: x['score'], reverse=True)[:top_k_final]


# 3. Hàm bọc (Wrapper) để thực thi lệnh truy xuất cho một sự kiện đơn lẻ.
def process_single_event(i, event_text):
    results = aic_retrieval_engine(
        event_text,
        use_quest=False,
        top_k_pinecone=4000,
        top_k_final=2000
    )
    return i, results

# Tách các truy vấn có cấu trúc (ví dụ: E1:, E2:) thành một danh sách các sự kiện độc lập theo trình tự thời gian.
def split_trake_query(query_text):
    if re.search(r'E1|\(1\)|1\.', query_text, re.IGNORECASE):
        try:
            parts = re.split(r'E\d+\s*|\(\d+\)\s*|\d+\.\s*', query_text, flags=re.IGNORECASE)
            context = (parts[0].split(":")[0] + " " if ":" in parts[0] else parts[0]).strip() + " "
            events = [context + a.strip(' ,;.:\n') for a in parts[1:] if a.strip()]
            if len(events) > 1: return events
        except: pass
    return [query_text]

def dp_solve(query_id, query_text, top_videos=30, lambda_penalty=0.00005):
    sub_events = split_trake_query(query_text)
    N = len(sub_events)
    event_frame_scores, vid_votes = {i: {} for i in range(N)}, {}

    # Thực thi đa luồng (Multithreading) để truy xuất dữ liệu vector cho tất cả các sự kiện (N) cùng lúc.
    with ThreadPoolExecutor(max_workers=min(N, 5)) as executor:
        futures = {executor.submit(process_single_event, i, evt): i for i, evt in enumerate(sub_events)}
        for future in as_completed(futures):
            i, evt_results = future.result()
            for res in evt_results:
                vid, n_val, score = res['video_id'], res['n'], res['score']
                if vid not in event_frame_scores[i]: event_frame_scores[i][vid] = {}
                event_frame_scores[i][vid][n_val] = score
                if vid not in vid_votes: vid_votes[vid] = set()
                vid_votes[vid].add(i)

    # Chấm điểm tổng hợp và lọc ra danh sách các video tiềm năng (candidate_vids) nhằm giảm thiểu không gian tìm kiếm cho thuật toán DP.
    vid_total_scores = {vid: (len(v_found), sum(max(event_frame_scores[i][vid].values()) for i in range(N) if vid in event_frame_scores[i])) for vid, v_found in vid_votes.items()}
    candidate_vids = sorted(vid_total_scores.keys(), key=lambda v: (vid_total_scores[v][0], vid_total_scores[v][1]), reverse=True)[:top_videos]
    final_sequences = []

    # Vòng lặp Quy hoạch động cho từng video ứng viên.
    for vid in candidate_vids:
        T_v = sorted({k for i in range(N) if vid in event_frame_scores[i] for k in event_frame_scores[i][vid].keys()})
        if len(T_v) < N: continue
        K = len(T_v)

        # Khởi tạo ma trận chi phí (dp) và ma trận lưu vết (trace).
        dp, trace = np.full((N, K), -np.inf), np.full((N, K), -1, dtype=int)

        # Điều kiện cơ sở (Base case) cho sự kiện đầu tiên.
        for t_idx, t in enumerate(T_v): dp[0, t_idx] = event_frame_scores[0].get(vid, {}).get(t, 0.0)

        # Bước tiến (State transition) với kỹ thuật tối ưu hóa độ phức tạp thời gian bằng biến running_max.
        for i in range(1, N):
            running_max, running_argmax = -float('inf'), -1
            for t_idx in range(1, K):
                t, t_prev = T_v[t_idx], T_v[t_idx - 1]
                val = dp[i-1, t_idx - 1] + lambda_penalty * t_prev
                if val > running_max: running_max, running_argmax = val, t_idx - 1

                score_it = event_frame_scores[i].get(vid, {}).get(t, 0.0)

                # Cập nhật điểm số hàm mục tiêu bao gồm: Điểm tại t + Điểm lớn nhất tích lũy - Hệ số phạt khoảng cách (lambda_penalty).
                dp[i, t_idx] = score_it + running_max - lambda_penalty * t
                trace[i, t_idx] = running_argmax

        # Dò ngược (Backtracking) để tái tạo chuỗi khung hình tối ưu từ ma trận lưu vết.
        best_last_t_idx = np.argmax(dp[N-1])
        best_vid_score = dp[N-1, best_last_t_idx]

        if best_last_t_idx != -1 and best_vid_score > -float('inf'):
            seq, curr_t, valid = [], best_last_t_idx, True
            for i in range(N-1, -1, -1):
                if curr_t == -1: valid = False; break
                seq.append(T_v[curr_t])
                curr_t = trace[i, curr_t]

            if valid:
                seq.reverse()
                # Ánh xạ chỉ số nội bộ (n) sang chỉ số khung hình thực tế (frame_idx) của video thông qua từ điển map_db.
                mapped_seq = [map_db.get(vid, {}).get(n, n) for n in seq]
                final_sequences.append({'video_name': vid, 'score': best_vid_score, 'frames': mapped_seq, 'n_frames': seq})

    # Sắp xếp lại danh sách các chuỗi tìm được theo điểm số giảm dần và trả về kết quả Top 100.
    final_sequences.sort(key=lambda x: x['score'], reverse=True)
    return final_sequences[:100]

In [4]:
# [BLOCK 4] RE-RANKING BẰNG VLM

import os, json
from PIL import Image

# Khai báo đường dẫn gốc trỏ tới kho dữ liệu khung hình (Keyframes) lưu trên Google Drive.
KEYFRAMES_DIR = "/content/drive/MyDrive/AIC26/Keyframes"

# Hàm tải ảnh tĩnh từ ổ đĩa.
# Sử dụng f"{int(n_val):04d}.jpg" để ép kiểu định dạng chuỗi 4 chữ số (ví dụ: 0001.jpg, 0123.jpg) nhằm đồng bộ tuyệt đối với chuẩn lưu trữ.
def load_frame_image(video_id, n_val):
    img_path = os.path.join(KEYFRAMES_DIR, video_id, f"{int(n_val):04d}.jpg")
    return Image.open(img_path) if os.path.exists(img_path) else None

def vlm_global_verifier(video_id, anchor_frames, sub_events):
    # Tải toàn bộ chuỗi hình ảnh tương ứng với chuỗi sự kiện.
    images = []
    for n in anchor_frames:
        img = load_frame_image(video_id, n)
        if img: images.append(img)

    if len(images) != len(sub_events): return 0.0

    prompt = f"""You are a RUTHLESS AI judge evaluating video retrieval. Look at these {len(images)} images in order.
    Do they EXACTLY match these events?
    """
    # Trình bày chuỗi sự kiện theo thứ tự thời gian.
    for idx, evt in enumerate(sub_events):
        prompt += f"Image {idx+1}: {evt}\n"

    # Áp đặt các quy tắc loại trừ nghiêm ngặt để hạn chế tối đa hiện tượng "ảo giác" của mô hình AI, bao gồm: kiểm tra nhận diện chữ (OCR), công cụ (Tools) và vật thể (Objects).
    prompt += """
    CRITICAL ELIMINATION RULES (Score 0.0 IMMEDIATELY if any rule is broken):
    1. OCR CHECK: Read ALL text/subtitles on the screen. If the text says "GIẤM" (vinegar) or something else, but the prompt asks for "NẤM" (mushroom), SCORE 0.0!
    2. TOOL CHECK: The action "Cắt" (cutting) REQUIRES a visible knife (dao) and usually a cutting board. If you see a person using a SPOON, a BLENDER, or pouring liquid, SCORE 0.0.
    3. OBJECT CHECK: You must clearly identify the specific object (mushroom, tofu, water chestnut).

    If it is a PERFECT match, score 1.0. Otherwise, score 0.0.
    Return ONLY JSON: {"confidence_score": }
    """

    # Truy vấn mô hình Ngôn ngữ - Hình ảnh (VLM).
    try:
        res = client.models.generate_content(
            model='gemini-3.1-pro',
            contents=images + [prompt],
            # Ép mô hình phải trả về định dạng chuẩn JSON để thuật toán dễ dàng bóc tách điểm số.
            config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0.0)
        )
        return float(json.loads(res.text.strip()).get('confidence_score', 0.0))
    except:
        return 0.0

In [5]:
# [BLOCK 5] THỰC THI CHUỖI TỔNG THỂ & XUẤT KẾT QUẢ CSV

import os, shutil, csv, time
from google.colab import files

trake_queries = {
    "query-p2-21-trake": "E1: Khoảnh khắc người đầu bếp bắt đầu trộn hỗn hợp nước xốt, trong nước xốt này có 2 thành phần là nước cam và vỏ cam.\nE2: Khoảnh khắc người đầu bếp bắt đầu cắt bỏ đầu tôm.\nE3: Người đầu bếp cho tôm ra dĩa. Hãy chọn khoảnh khắc con tôm đầu tiên chạm vào dĩa.\nE4: Khoảnh khắc tép cam thứ 4 được xếp lên dĩa khi trang trí món ăn.",
}
# Khởi tạo môi trường lưu trữ kết quả.
# Hệ thống sẽ tự động dọn dẹp thư mục cũ (nếu có) để tránh xung đột dữ liệu (data conflict) từ các lần chạy thực nghiệm trước đó.
SUBMISSION_DIR = "trake_submission"
if os.path.exists(SUBMISSION_DIR): shutil.rmtree(SUBMISSION_DIR)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

USE_VLM = True
TOP_K_VLM_CANDIDATES = 30    # Chỉ giới hạn chấm điểm VLM cho top 30 video tiềm năng nhất để tối ưu thời gian.

# Vòng lặp thực thi cốt lõi: Xử lý tuần tự từng truy vấn có trong bộ dữ liệu.
for q_id, q_text in trake_queries.items():

    # Kích hoạt thuật toán Quy hoạch động để trích xuất danh sách 100 video có điểm số không gian gần nhất.
    dp_results = dp_solve(q_id, q_text, top_videos=100, lambda_penalty=0.00005)
    sub_events = split_trake_query(q_text)
    final_list = dp_results.copy()

    # Tiến trình Đánh giá lại (Re-ranking) bằng VLM (nếu USE_VLM = True).
    if USE_VLM and len(final_list) > 0:
        for rank, candidate in enumerate(final_list[:TOP_K_VLM_CANDIDATES]):
            vid = candidate['video_name']

            # Trích xuất chuỗi khung hình nội bộ (n_frames) để tải ảnh tương ứng.
            n_seq = candidate.get('n_frames', candidate.get('frames', []))

            # Gọi mô hình VLM chấm điểm độ khớp ngữ nghĩa thực tế.
            vlm_score = vlm_global_verifier(vid, n_seq, sub_events)
            candidate['vlm_score'] = vlm_score

            # Cơ chế Dừng sớm (Early Stopping): Nếu phát hiện video khớp hoàn hảo (điểm >= 0.95), ngắt vòng lặp ngay lập tức để bảo toàn kết quả và tiết kiệm API Quota.
            if vlm_score >= 0.95:
                break

            # Dừng 2 giây để tránh lỗi vượt quá giới hạn tần suất yêu cầu (Rate Limit) của máy chủ API.
            time.sleep(2)

        # Cập nhật lại thứ hạng (Re-rank) cho tập TOP_K ứng viên dựa trên điểm số VLM mới, các video thuộc top sau (từ 31-100) vẫn giữ nguyên vị trí cũ.
        top_k_part = final_list[:TOP_K_VLM_CANDIDATES]
        top_k_part.sort(key=lambda x: x.get('vlm_score', -1.0), reverse=True)
        final_list = top_k_part + final_list[TOP_K_VLM_CANDIDATES:]

    # Ghi xuất kết quả cuối cùng (Top 100) ra tệp định dạng CSV.
    if len(final_list) > 0:
        with open(f"{SUBMISSION_DIR}/{q_id}.csv", 'w', newline='') as f:
            writer = csv.writer(f)
            for res in final_list[:100]:
                writer.writerow([res['video_name']] + res['frames'])

# Đóng gói (Archive) toàn bộ thư mục CSV thành một tệp nén (.zip) duy nhất.
shutil.make_archive("submission", 'zip', root_dir=SUBMISSION_DIR)

# Kích hoạt lệnh giao tiếp với trình duyệt để tự động tải tệp nén xuống hệ thống cục bộ.
files.download("submission.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>